In [1]:
import pandas as pd
import random
import os

import torch
from unsloth import FastLanguageModel, get_chat_template
import re
import gc
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import accelerate

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\PokeAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 22:20:01.036000 8396 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


# Padronizando o df para o formato do train

In [5]:
model_path = "C:/PokeAI/Models/gemma-3-4b-it"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    device_map="auto",
)

dataset = pd.read_csv('C:/PokeAI/databases/Perguntas_Metagame.csv')

dataset['messages'] = dataset.apply(
    lambda row: [
        {'role': 'user', 'content': str(row['pergunta'])},
        {'role': 'assistant', 'content': str(row['resposta'])}
    ],
    axis=1
)

dataset['text'] = dataset['messages'].apply(
    lambda messages: tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
)

dataset = Dataset.from_pandas(
    dataset[['text']],
    preserve_index=False
)


==((====))==  Unsloth 2026.9.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660 SUPER. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights: 100%|██████████| 883/883 [00:43<00:00, 20.45it/s] 


# Aplicando o teste de Train no **Gemma**

In [ ]:
model_path = "C:/PokeAI/Models/gemma-3-4b-it"

# Carregamento do modelo
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    device_map="auto",
)

# Preparação do LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Configuração do treinamento
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="C:/PokeAI/results/gemma",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        weight_decay=0.01,
        logging_steps=1,
        fp16=True,
        report_to="none",
        max_length=2048,
    ),
)

# Treinamento
trainer.train()

# Histórico do treinamento
history = trainer.state.log_history

# Transformando o histórico em DataFrame
df_history = pd.DataFrame(history)

output_path = "C:/PokeAI/Models/gemma-3-4b-it-finetuned"

model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
df_history.to_csv(f"C:/PokeAI/Models/testes/logs_train/training_history_gemma.csv", index=False)

==((====))==  Unsloth 2026.9.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660 SUPER. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Current model requires 34818 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 883/883 [01:25<00:00, 10.28it/s] 


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]: 100%|██████████| 121/121 [00:00<00:00, 279.93 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 121 | Num Epochs = 1 | Total steps = 16
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)


Step,Training Loss
1,6.394318
2,6.207489
3,5.997942
4,4.733479
5,4.669155
6,4.380558
7,3.979923
8,3.972613
9,3.639881
10,3.757346


Unsloth: Restored added_tokens_decoder metadata in C:/PokeAI/results/gemma\checkpoint-16\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in C:/PokeAI/results/gemma\checkpoint-16.


# Aplicando o Train em todos juntos

In [ ]:
for modelo in ['gemma-3-4b-it', 'Llama-3.2-3B-Instruct', 'DeepSeek-V4-Pro-Qwen3.5-4B', 'Qwen3.5-4B']:
    model_path = f"C:/PokeAI/Models/{modelo}"

    # Carregamento do modelo
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=True,
        device_map="auto",
    )

    # Preparação do LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    # Configuração do treinamento
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        args=SFTConfig(
            output_dir=f"C:/PokeAI/results/{modelo}",
            num_train_epochs=1,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            weight_decay=0.01,
            logging_steps=1,
            fp16=True,
            report_to="none",
            max_length=2048,
        ),
    )

    # Treinamento
    trainer.train()

    # Histórico do treinamento
    history = trainer.state.log_history

    # Transformando o histórico em DataFrame
    df_history = pd.DataFrame(history)

    output_path = f"C:/PokeAI/Models/{modelo}-finetuned"

    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    df_history.to_csv(f"C:/PokeAI/Models/testes/logs_train/training_history_{modelo}.csv", index=False)